In [ ]:
import os
import glob
import numpy as np
from multiprocessing import Pool

from tqdm.notebook import tqdm
import numpy as np
from tqdm import tqdm

from lmsiage.zarr_index_tools import files_with_array, update_index_for_dir
from lmsiage.grid_age import GridAge

from lmsiage.mesh_file import MeshFile
import matplotlib.pyplot as plt

In [ ]:
force = True
sia_dir = 'NERSC_arctic25km_sea_ice_age_DA'
mesh_dir = f'{sia_dir}/mesh'
age_dir = f'{sia_dir}/age_da_da5l'
unc_dir = None
grid_dir = f'NERSC_arctic25km_sea_ice_age_DA/grid'

mesh_init_file = 'mesh_arctic_ease_25km_max7.npz'
xc = np.load(mesh_init_file)['xc']
yc = np.load(mesh_init_file)['yc'][::-1]
mask = np.load(mesh_init_file)['mask']
xgrd, ygrd = np.meshgrid(xc, yc)

sic = np.load('OSISAF_ice_drift_CDR_postproc/1991/ice_drift_nh_ease2-750_cdr-v1p0_24h-199101011200.nc.npz')['c']
landmask = np.isnan(sic).astype(float)
landmask[landmask == 0] = np.nan

In [ ]:
start_date = '20120915'
stop_date = '20251231'

age_files = glob.glob(f'{age_dir}/*/*zip')
age_files_date = []
basename = None
for f in age_files:
    basename = os.path.basename(f)
    if start_date <= basename.split('_')[1].split('.')[0] <= stop_date:
        age_files_date.append(f)

print(len(age_files_date), age_files_date[0], age_files_date[-1])

In [ ]:
grid_age = GridAge(mesh_dir, age_dir, unc_dir, grid_dir, xgrd, ygrd, mask, force)
for unc_file in tqdm(age_files_date):
    grid_age(unc_file)


In [ ]:
grid_dir21 = '/Volumes/sim/data/NERSC_arctic25km_sea_ice_age_v2p1/grid'
year = 2025
mmdd = '1231'
mesh_file = f'{grid_dir21}/{year}/grid_{year}{mmdd}.zip'

print(MeshFile(mesh_file).read_names())

age, sic_1y, sic_2y = MeshFile(mesh_file).load(['age', 'sic_1yi', 'sic_2yi'], as_dict=False)

c0, c1 = 100, 300
r0, r1 = 100, 320
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(age[r0:r1, c0:c1], clim=[0, 2])
axs[1].imshow(sic_1y[r0:r1, c0:c1], clim=[0, 100])
axs[2].imshow(sic_2y[r0:r1, c0:c1], clim=[0, 100])
for ax in axs:
    ax.set_aspect('equal')
    plt.colorbar(ax.images[0], ax=ax, shrink = 0.5)
plt.show()

In [ ]:
mesh_file = f'{grid_dir}/{year}/grid_{year}{mmdd}.zip'

print(MeshFile(mesh_file).read_names())

age, sic_1y, sic_2y = MeshFile(mesh_file).load(['age', 'sic_1yi', 'sic_2yi'], as_dict=False)

fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(age[r0:r1, c0:c1], clim=[0, 2])
axs[1].imshow(sic_1y[r0:r1, c0:c1], clim=[0, 100])
axs[2].imshow(sic_2y[r0:r1, c0:c1], clim=[0, 100])
for ax in axs:
    ax.set_aspect('equal')
    plt.colorbar(ax.images[0], ax=ax, shrink = 0.5)
plt.show()